In [32]:
### IMPORT EXCEPTION MODULES
from requests.exceptions import Timeout
from github import GithubException, UnknownObjectException, IncompletableObject

### IMPORT SYSTEM MODULES
from github import Github
import os, logging, pandas, csv, tempfile, shutil
from datetime import datetime, timezone, timedelta
from tqdm import tqdm
from pathlib import Path
import numpy as np

from truckfactor.compute import main as compute_tf
import re
import unicodedata
from typing import Iterable, Tuple, List, Set, Dict, List
from collections import Counter

### IMPORT CUSTOM MODULES
import sys
sys.path.append('../')
import Settings as cfg
import Utilities as util
import subprocess, tempfile, shutil

from git import Repo, exc as git_exc
import time

# NEW BREAK IDENTIFICATION 
                                         (MADE BY SAM UTZ)

In [2]:
def get_NONCODING(folder: str, dev_login: str) -> pandas.DataFrame:

    """
    Build the developer's DAILY 'other-actions' table.
    Returns a dataframe whose index is the *action*
    ('issues/pull_requests', 'issues_comments', …) and whose
    columns are day-strings.
    """
    files = {
    "prs": (
        "prs_repo.csv",
        {"PR_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "prs_comments": (
        "prs_comments.csv",
        {"comment_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues": (
        "issues_repo.csv",
        {"issue_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues_comments": (
        "issues_comments_repo.csv",
        {"comment_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues_events": (
        "issues_events_repo.csv",
        {"event_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues_timeline": (
        "issues_timeline_repo.csv",
        {"event_id": "id", "created_at": "date", "created_by": "creator_login"},
    )
    }

    # ---------- read / filter every file ----------
    dfs = {}
    for key, (fname, rename_map) in files.items():
        dfs[key] = _load_activity_csv(folder, fname, rename_map, dev_login)

    # ---------- split issues vs PRs -------------
    # Old logic: issues endpoint also returns PRs; remove rows whose id
    # matches a PR id so we don’t double-count.
    if not dfs["issues"].empty and not dfs["prs"].empty:
        dfs["issues"] = dfs["issues"][~dfs["issues"].id.isin(dfs["prs"].id)]

    # ---------- build the day range -------------
    # Derive it from the *actual* activity we just read.
    #
    # 1) gather every non-empty dataframe
    non_empty = [df for df in dfs.values() if not df.empty]

    if non_empty:
        # 2) earliest / latest date across *all* action types
        min_date = min(df["date"].min() for df in non_empty)
        max_date = max(df["date"].max() for df in non_empty)
    else:
        # Developer has no activity at all → default to one-day range
        min_date = max_date = pandas.Timestamp.today()

    # 3) full, dense list of day strings
    day_cols = (
        pandas.date_range(
            start=pandas.to_datetime(min_date).normalize(),
            end=pandas.to_datetime(max_date).normalize(),
            freq="D",
        )
        .strftime("%Y-%m-%d")
        .tolist()
    )

    # ---------- helper to create one timeline row ----------
    def _timeline_row(action_name, df_raw):
        row = [action_name]
        if df_raw.empty:
            row += [0] * len(day_cols)
            return row
        counts = (
            pandas.to_datetime(df_raw["date"])
            .dt.date
            .value_counts()
            .to_dict()
        )
        for d in day_cols:
            row.append(counts.get(pandas.to_datetime(d).date(), 0))
        return row

    # ---------- compile all action rows ----------
    rows = []
    if not dfs["issues"].empty:
        rows.append(_timeline_row("issues", dfs["issues"]))
    if not dfs["issues_comments"].empty:
        rows.append(_timeline_row("issues_comments", dfs["issues_comments"]))
    if not dfs["issues_events"].empty:
        rows.append(_timeline_row("issues_events", dfs["issues_events"]))
    if not dfs["prs"].empty:
        rows.append(_timeline_row("pull_requests", dfs["prs"]))
    if not dfs["prs_comments"].empty:
        rows.append(_timeline_row("pull_requests_comments", dfs["prs_comments"]))

    # (commits are already encoded in coding_history_table, so we skip them here)

    actions = pandas.DataFrame(rows, columns=["action"] + day_cols).set_index("action")

    return actions

def _load_activity_csv(folder: str,
                       filename: str,
                       rename_map: Dict[str, str],
                       dev_login,
                       usecols: list[str] = None,
                       ) -> pandas.DataFrame:
    """
    Read *filename* in *folder*, rename to the canonical columns
    ('id','date','creator_login'), keep ONLY the specified dev, and
    return three columns.  On any problem → empty df.
    """
    path = os.path.join(folder, filename)
    try:
        df = pandas.read_csv(path, sep=cfg.CSV_separator, usecols=usecols)
    except FileNotFoundError:
        logging.info("File %s not found – skipping", path)
        return pandas.DataFrame(columns=["id", "date", "creator_login"])
    except Exception as e:
        logging.warning("Could not read %s: %s", path, e)
        return pandas.DataFrame(columns=["id", "date", "creator_login"])

    df = df.rename(columns=rename_map)
    # keep only the columns we need, ignore anything extra
    df = df[["id", "date", "creator_login"]]
    df = df[df.creator_login == dev_login]
    # allow str OR list[str]
    if isinstance(dev_login, list):
        df = df[df.creator_login.isin(dev_login)]
    else:
        df = df[df.creator_login == dev_login]
    return df.reset_index(drop=True)

def get_ACTIVITY(folder: str, dev: str) -> pandas.DataFrame:

    path = os.path.join(folder, "commit_list.csv")

    # ─── load & clean ──────────────────────────────────────────────────
    df = pandas.read_csv(path, sep=cfg.CSV_separator, parse_dates=["created_at"])

    df = df[df["author_id"] == dev]           # keep only this dev
    if df.empty:                                    # no commits at all
        raise ValueError(f"No commits found for {dev}")

    dates = df["created_at"].dt.normalize()
    if getattr(dates.dt, "tz", None) is not None:
            dates = dates.dt.tz_localize(None)
    # ─── per-day aggregation ──────────────────────────────────────────
    daily_counts = (
        df.groupby(df["created_at"].dt.normalize())       # strip and remove hh:mm:ss 
          .size()
          .rename("commits")
    )

    daily_counts = (
        dates.value_counts()
             .sort_index()
             .rename("commits")
             .astype("int64")
    )
    daily_counts.index.name = "date"
    return daily_counts.to_frame()

def get_timeline(folder: str, dev: str) -> pandas.DataFrame:
    """ActivitiesExtractor.py
    This function will take all the activity and make a daily aggregated df
    """
    # actions: rows = action names, cols = day strings "YYYY-MM-DD"
    actions = get_NONCODING(folder, dev)

    actions = actions.transpose()  # make the index a DatetimeIndex
    
    # make the index a DatetimeIndex (tz-naive, normalized)
    actions.index = pandas.to_datetime(actions.index, errors="coerce").tz_localize(None)
    actions.index.name = "date"

    # commits: DataFrame with index=date (DatetimeIndex), col 'commits'
    commits = get_ACTIVITY(folder, dev)

    # union the date ranges and align
    idx = actions.index.union(commits.index)
    actions = actions.reindex(idx, fill_value=0)
    commits   = commits.reindex(idx, fill_value=0)

    # merge
    df = actions.join(commits, how="outer")
    df = df.fillna(0)

    # ensure expected columns exist even if that action never occurred
    for col in ["pull_requests", "issues", "issues_comments", "issues_events", "pull_requests_comments"]:
        if col not in df.columns:
            df[col] = 0

    # booleans
    df["coding_day"] = (df["commits"] > 0) | (df["pull_requests"] > 0)
    df["nc_day"] = (df[["issues", "issues_comments", "issues_events", "pull_requests_comments"]].sum(axis=1) > 0)

    # nice ordering
    cols = ["commits", "pull_requests", "issues", "issues_comments", "issues_events", "pull_requests_comments",
            "coding_day", "nc_day"]
    # keep any extra columns too (if you later add more)
    df = df[[c for c in cols if c in df.columns] + [c for c in df.columns if c not in cols]]
    return df.sort_index()


In [3]:
def write_pauses_table(
        df: pandas.DataFrame,
        out_path: os.PathLike,
        authors: list[str] | None = None,
        *,
        user_col: str = "author_id",
        date_col: str = "created_at",
        tail_to_today: bool = False
    ) -> pandas.DataFrame:

    df[date_col] = pandas.to_datetime(df[date_col]).dt.normalize()

    if authors is None:
        authors = df[user_col].unique()

    rows = []
    
    count =0
    for dev in authors:
        user_df = df[df[user_col] == dev]
        pause_len_1 = len(user_df[date_col].dt.date.unique())
        pause_len_2 =len(user_df)
        if user_df.empty:
            continue

        active_days = sorted(user_df[date_col].dt.date.unique())
        current_row = [dev]

        for i in range(len(active_days) - 1):
            prev_day = active_days[i]
            next_day = active_days[i + 1]
            gap = (next_day - prev_day).days
            if gap > 1:
                # Inactivity starts the day after prev_day
                current_row.append(f"{(prev_day + pandas.Timedelta(days=1)).strftime('%Y-%m-%d')}/{next_day.strftime('%Y-%m-%d')}")
            else:
                count += 1


        if tail_to_today and active_days:
            today = _date.today()
            gap = (today - active_days[-1]).days
            if gap > 1:
                current_row.append(f"{active_days[-1]}/{today}")

        if len(current_row) > 1:
            rows.append(current_row)
    
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", newline="",encoding="utf-8" ) as f:
        csv.writer(f, delimiter=",", quoting=csv.QUOTE_NONE).writerows(rows)

    return pandas.DataFrame(rows)

def get_commit_based_core_devs(commits, threshold=0.8):
    """
    commits: List[dict] where each dict contains at least the 'author' key.
    Example: [{'author': 'alice'}, {'author': 'bob'}, {'author': 'alice'}, ...]

    Returns: List of core developers (author names) who together authored >= threshold of commits.
    """
    # Count commits per developer
    author_commit_counts = Counter(commit["author_id"] for commit in commits)

    # Sort developers by number of commits (descending)
    sorted_authors = author_commit_counts.most_common()

    total_commits = sum(author_commit_counts.values())
    cumulative = 0
    core_devs = []

    for author, count in sorted_authors:
        cumulative += count
        core_devs.append(author)
        if cumulative / total_commits >= threshold:
            break

    return core_devs

def identifyInactivityPeriods(organizationFolder, organization, project):
    """Identifies the inactivity periods of the developers in the organization"""
    #url = "https://github.com/" + organization + "/" + project + ".git"
    #authors, emails = findCoreDevelopers(url, name=project)
    
    organizationFolder = organizationFolder + "/" + organization + "/" + project

    commits =  pandas.read_csv(organizationFolder + "/commit_list.csv", parse_dates=["created_at"], encoding="utf-8", header=0, sep=cfg.CSV_separator)

    commit_authors = get_commit_based_core_devs(commits.to_dict(orient='records'))

    pauses = write_pauses_table(commits, organizationFolder + "/pauses_commits.csv", commit_authors, user_col = "author_id", date_col="created_at")
    return commit_authors, pauses

def getFarOutThreshold(values, dev): ### If it is satisfying, move the function into UTILITIES
    import numpy
    th = 0
    q_3rd = numpy.percentile(values,75)
    q_1st = numpy.percentile(values,25)
    iqr = q_3rd-q_1st
    if iqr > 1:
        th = q_3rd + 3*iqr
    return th

def addToBreaksList(pauses, currentBreaks, th):
    for _, p in pauses.iterrows():
        if (p['len'] > th) and (p['dates'] not in currentBreaks.dates.tolist()):
            util.add(currentBreaks, [p['len'], p['dates'], th])
    return currentBreaks

def cleanClearBreaks(clearBreaks, breaks):
    for _, b in breaks.iterrows():
        clearBreaks = clearBreaks[clearBreaks.dates != b['dates']] # If it was in the long_breaks list, remove ot from there
    return clearBreaks

def identifyBreaks(pauses_dates_list, dev, window, shift,
                   debug_folder=None):           # NEW ARG
    '''
    Removes SURE BREAKS from windows to calculate Tfov
    and — with debug_folder — writes a per-window diagnostics CSV.
    '''
    breaks_df = pandas.DataFrame(columns=['len', 'dates', 'th'])
    diagnostics = []                             # NEW
    count = 0
    for row in pauses_dates_list:
        if row[0] != dev:              # ⬅️  ignore other developers
            continue
        
        count += 1
        if count % 50 == 0:  # Print progress every 100 developers
            print(count)
        intervals_list = [ x for x in row[1:]
                          if isinstance(x, str) and '/' in x and x.strip()]
        
        intervals_list.sort(key=lambda s: s.split('/')[0])

        if not all(a.split('/')[0] <= b.split('/')[0]
                for a, b in zip(intervals_list, intervals_list[1:])):
            print("⚠️  intervals_list UNSORTED for", dev)

        if not intervals_list:
            print(dev, 'has NO valid pauses')
            continue                      # <- don’t bail out; just skip

        clear_breaks = pandas.DataFrame(columns=['len', 'dates'])

        FPS_dt = datetime.strptime(intervals_list[0].split('/')[0], '%Y-%m-%d')
        LPE_dt = datetime.strptime(intervals_list[-1].split('/')[1], '%Y-%m-%d')

        win_start, win_end = FPS_dt, FPS_dt + timedelta(days=window)
        last_th = 0
        while win_end < LPE_dt:
            win_pauses_list = pandas.DataFrame(columns=['len', 'dates'])
            partially_included_pauses_list = pandas.DataFrame(columns=['len', 'dates'])

            for interval in intervals_list:
                int_start_str, int_end_str = interval.split('/')          # keep strings
                int_start_dt  = datetime.strptime(int_start_str, '%Y-%m-%d')
                int_end_dt    = datetime.strptime(int_end_str,   '%Y-%m-%d')
                pause_len = util.daysBetween(int_start_str, int_end_str)
                # fully inside
                if int_start_dt >= win_start and int_end_dt <= win_end:
                    util.add(win_pauses_list, [pause_len, interval])
                # touches boundary
                if ((int_start_dt <= win_end and int_end_dt > win_end) or
                    (int_end_dt >= win_start and int_start_dt < win_start)):
                    util.add(partially_included_pauses_list, [pause_len, interval])

            win_pauses = len(win_pauses_list)
            pauses = pandas.concat([win_pauses_list,
                                    partially_included_pauses_list],
                                    ignore_index=True)

            # --- decision logic (unchanged) ---------------------------------
            win_th = None
            added_flag = False
            if win_pauses >= 4:
                win_th = getFarOutThreshold(win_pauses_list['len'], dev)
                if win_th > 0:
                    before = len(breaks_df)
                    breaks_df = addToBreaksList(pauses, breaks_df, win_th)
                    added_flag = len(breaks_df) > before
                    last_th = win_th
                elif last_th > 0:
                    before = len(breaks_df)
                    breaks_df = addToBreaksList(pauses, breaks_df, last_th)
                    added_flag = len(breaks_df) > before
            else:
                if last_th > 0:
                    before = len(breaks_df)
                    breaks_df = addToBreaksList(pauses, breaks_df, last_th)
                    added_flag = len(breaks_df) > before

                clear_breaks = cleanClearBreaks(clear_breaks, breaks_df)
                for _, p in pauses.iterrows():
                    if (p['len'] >= window and
                        p['dates'] not in clear_breaks.dates.tolist() and
                        p['dates'] not in breaks_df.dates.tolist()):
                        util.add(clear_breaks, p)

            # ----------- NEW: record diagnostics for this window -------------
            diagnostics.append({
                'win_start': win_start.date(),
                'win_end':   win_end.date(),
                'win_pauses': win_pauses,
                'pause_lengths': ';'.join(map(str, win_pauses_list['len'].tolist())),
                'partial_lengths': ';'.join(map(str, partially_included_pauses_list['len'].tolist())),
                'win_th': win_th,
                'last_th': last_th,
                'added_as_break': 'yes' if added_flag else 'no'
            })
            # -----------------------------------------------------------------

            win_start += timedelta(days=shift)
            win_end   = win_start + timedelta(days=window)


    return breaks_df

### LABEL BREAKS

In [39]:
def label_developers_activity() -> pandas.DataFrame:
    """
    main function for labeling developers

    sets up varables to call label_timeline

    """
    #identifyInactivityPeriods
    #"Resources/repositories.txt"
    repos_file= '../' + cfg.repos_file
    #"../Organizations"
    organizationFolder = cfg.main_folder


    #identifyBreaks
    win = cfg.sliding_window_size
    shift = cfg.shift

    with open(repos_file) as f:
        repos_line = f.readlines()
        for repo in repos_line:
            #take the end '\n' out
            repo = repo.rstrip('\n')
            organization, project = repo.split('/')

            print(f"Start Identifying inactivity periods for {organization}/{project}...")

            authors, pauses = identifyInactivityPeriods( organizationFolder, organization, project)
            #make pauses to a csv file at this location C:\Users\samut\OneDrive\Documents\GitHub\developersInactivityAnalysisCOPY\Organizations\Rdatatable\data.table\Results
        
            pauses_list = pauses.values.tolist()
            print(f"{len(authors)} Developers inactivity periods identified")

            output_folder = organizationFolder + '/' + repo + "/Results"
            os.makedirs(output_folder, exist_ok=True)
            
            for dev in authors:
                timeline_folder = organizationFolder + '/' + repo + '/' + cfg.timeline_folder_name
                os.makedirs(timeline_folder, exist_ok=True)
            
                timeline_path = Path(timeline_folder) / f"{dev}_timeline.csv"

                if timeline_path.is_file():
                    user_timeline = pandas.read_csv(timeline_path, sep=cfg.CSV_separator, index_col=0)
                else:
                    folder = organizationFolder + '/' + repo
                    user_timeline = get_timeline(folder, dev)

                    #transpose the user_timeline making the frist row the first column
                    user_timeline.to_csv(timeline_path, sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, quoting=None, lineterminator='\n')

                print(f"{dev}")
            
                #make a break folder 
                breaks_folder = organizationFolder + '/' + repo + "/Breaks"
                os.makedirs(breaks_folder, exist_ok=True)
        
                breaks_path =  Path(breaks_folder)/  f"{dev}_breaks.csv"              

                if breaks_path.is_file():
                    breaks_df = pandas.read_csv(breaks_path, sep=cfg.CSV_separator, index_col=0)  

                else:
                    breaks_df = pandas.DataFrame(columns=['len', 'dates', 'th'])
                    breaks_df = identifyBreaks(pauses_list, dev=dev, window=win, shift=shift, debug_folder=output_folder )
                    breaks_df.to_csv(breaks_path, sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, index=False, lineterminator="\n")
                            
                #add label_timeline
                user_timeline = label_timeline(user_timeline, breaks_df)


                out_csv = Path(output_folder) / f"{dev}_labeled_timeline.csv"
                user_timeline.to_csv(out_csv,
                                      sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, quoting=None, lineterminator='\n')
label_developers_activity()

Start Identifying inactivity periods for Rdatatable/data.table...
6 Developers inactivity periods identified
mattdowle


C:\Users\samut\AppData\Local\Temp\ipykernel_15848\2845437192.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)


MichaelChirico


C:\Users\samut\AppData\Local\Temp\ipykernel_15848\2845437192.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)


jangorecki


C:\Users\samut\AppData\Local\Temp\ipykernel_15848\2845437192.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)


arunsrinivasan


C:\Users\samut\AppData\Local\Temp\ipykernel_15848\2845437192.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)


ben-schwen
tdhock


C:\Users\samut\AppData\Local\Temp\ipykernel_15848\2845437192.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)
C:\Users\samut\AppData\Local\Temp\ipykernel_15848\2845437192.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)


Treats all non-break rows as ACTIVE

In [37]:
def label_timeline(user_timeline, breaks_df):
    """
    Make a labled timeline of devlopers breaks

    given a user_timeline and breaks_df
    user_timeline
    commits,pull_requests,issues,issues_comments,issues_events,pull_requests_comments,coding_day,nc_day
    0,0,2,1,0,0,False,True
    3,0,0,0,0,0,True,False
    0,0,0,0,0,0,False,False

    breaks_df
    len,dates,th
    72,2015-05-05/2015-07-16,59.25
    """
    df = user_timeline.copy()

    for breaks in breaks_df.itertuples():
        start = breaks.dates.split('/')[0]
        end = breaks.dates.split('/')[1]

        break_range = pandas.date_range(start=pandas.to_datetime(start)+pandas.Timedelta(days=1),
                                end=pandas.to_datetime(end)-pandas.Timedelta(days=1))

        #from the start to end of 
        for date in break_range:
            date = date.strftime("%Y-%m-%d")
            df.at[date, "break_day"] = True
            df.at[date, "th"] = breaks.th
            df.at[date, "len"] = breaks.Index

    for col in ["coding_day", "nc_day", "break_day"]:
        df[col] = df[col].fillna(False).astype(bool)    
        
    # Ensure a proper DatetimeIndex and chronological order
    df.index = pandas.to_datetime(df.index)
    df = df.sort_index()

    # Optional: unmark the break *end* day (commit day) so it’s not counted as break
    # This matches your example where the end day is ACTIVE with a transition note.
    for breaks in breaks_df.itertuples():
        end = pandas.to_datetime(breaks.dates.split('/')[1])
        if end in df.index:
            df.at[end, "break_day"] = False

    # Default Tfov if not carried on the rows
    if "th" in df.columns and df["th"].notna().any():
        tfov_default = int(round(df["th"].dropna().median()))
    elif "th" in breaks_df.columns and breaks_df["th"].notna().any():
        tfov_default = int(round(breaks_df["th"].dropna().median()))
    else:
        tfov_default = 30  # conservative fallback

    gone_days = 365

    # Convenience flags
    df["event_day"] = df["coding_day"] | df["nc_day"]

    # Start with ACTIVE everywhere; we’ll overwrite break ranges next
    df["state"] = "ACTIVE"
    df["transition"] = ""

    # Identify contiguous break windows (groups of consecutive True in break_day)
    bd = df["break_day"]
    group_id = (bd != bd.shift(1)).cumsum()
    break_groups = [g for g, flag in df.groupby(group_id) if flag["break_day"].iloc[0]]

    # Precompute last event BEFORE a given date (global, across timeline)
    all_events_idx = df.index[df["event_day"]]

    def last_event_before(ts):
        # Return the latest event date strictly before ts, or None
        idx = all_events_idx[all_events_idx < ts]
        return idx.max() if len(idx) else None

    for gid, block in df.groupby(group_id):
        if not block["break_day"].iloc[0]:
            continue  # not a break chunk

        # This is one contiguous break [start .. end] (inclusive)
        start_ts = block.index[0]
        end_ts   = block.index[-1]

        # Tfov for this window (take the first non-na th inside; else default)
        th_vals = df.loc[start_ts:end_ts, "th"].dropna()
        tfov_days = int(round(th_vals.iloc[0])) if len(th_vals) else tfov_default

        # Lookahead info: is there any non-coding event in this break?
        has_nc = bool((df.loc[start_ts:end_ts, "nc_day"]).any())
        first_nc_ts = df.loc[start_ts:end_ts].index[df.loc[start_ts:end_ts, "nc_day"]].min() if has_nc else None

        # Anchor silence to the last event (coding or non-coding) before the break starts
        anchor = last_event_before(start_ts)

        # Track the most recent non-coding event seen (within this break) to measure silence
        last_nc = None

        # Walk day by day inside the break
        for d in block.index:
            coding = bool(df.at[d, "coding_day"])
            nc     = bool(df.at[d, "nc_day"])

            # If coding somehow appears inside a break, force ACTIVE
            if coding:
                df.at[d, "state"] = "ACTIVE"
                continue

            # Non-coding event day => NON_CODING and update last_nc
            if nc:
                df.at[d, "state"] = "NON_CODING"
                last_nc = d
                continue

            # Silent day inside a break -> decide via Tfov and gone
            # Compute silence since the most relevant last event:
            # - Prefer last NC inside break; else use last event before break; else start-of-break as approximate anchor.
            ref = last_nc if last_nc is not None else (anchor if anchor is not None else start_ts - pandas.Timedelta(days=1))
            silent = (d - ref).days

            if silent > gone_days:
                df.at[d, "state"] = "GONE"
            elif silent > tfov_days:
                df.at[d, "state"] = "INACTIVE"
            else:
                # If this break contains (or will contain) any NC, keep it NON_CODING up to tfov since last_nc/anchor.
                if has_nc:
                    # Before the first NC happens, include those early days as NON_CODING if still within Tfov
                    df.at[d, "state"] = "NON_CODING"
                else:
                    # No NC at all in this break: stay ACTIVE until Tfov is exceeded
                    df.at[d, "state"] = "ACTIVE"

    # Transitions: back-to-coding, reactivation, comeback (based on state change and event on the day)
    prev_state = None
    for d in df.index:
        st = df.at[d, "state"]
        if prev_state is not None and st != prev_state:
            # Only label transition if an event actually occurs on this day
            if df.at[d, "event_day"]:
                if prev_state == "NON_CODING" and df.at[d, "coding_day"]:
                    df.at[d, "transition"] = "BACK_TO_CODING"
                elif prev_state == "INACTIVE" and (df.at[d, "coding_day"] or df.at[d, "nc_day"]):
                    df.at[d, "transition"] = "REACTIVATION"
                elif prev_state == "GONE" and (df.at[d, "coding_day"] or df.at[d, "nc_day"]):
                    df.at[d, "transition"] = "COMEBACK"
        prev_state = st

    return df


Algo B (with NCUT) → “Legacy break-gated view.”
Treats break windows as the only places where “real” inactivity/non-coding labeling happens. Outside breaks it collapses almost everything non-coding-or-silent into NCUT (i.e., “not a break”). It mirrors the old splitBreak vibe.

In [ ]:
def label_timeline(user_timeline, breaks_df):
    """
    Make a labled timeline of devlopers breaks

    given a user_timeline and breaks_df
    user_timeline
    commits,pull_requests,issues,issues_comments,issues_events,pull_requests_comments,coding_day,nc_day
    0,0,2,1,0,0,False,True
    3,0,0,0,0,0,True,False
    0,0,0,0,0,0,False,False

    breaks_df
    len,dates,th
    72,2015-05-05/2015-07-16,59.25
    """
    df = user_timeline.copy()

    for breaks in breaks_df.itertuples():
        start = breaks.dates.split('/')[0]
        end = breaks.dates.split('/')[1]

        break_range = pandas.date_range(start=pandas.to_datetime(start)+pandas.Timedelta(days=1),
                                end=pandas.to_datetime(end)-pandas.Timedelta(days=1))

        #from the start to end of 
        for date in break_range:
            date = date.strftime("%Y-%m-%d")
            df.at[date, "break_day"] = True
            df.at[date, "th"] = breaks.th
            df.at[date, "len"] = breaks.Index

    for col in ["coding_day", "nc_day", "break_day"]:
        df[col] = df[col].fillna(False).astype(bool)    

    state = pandas.Series(index=df.index, dtype="object")

    state[df["coding_day"]] = "ACTIVE"

    outside = (~df["break_day"]) & (~df["coding_day"])
    state[outside] = "NCUT"

    starts = (df["break_day"] & ~df["break_day"].shift(fill_value=False)).cumsum()
    B_all = df[df["break_day"]].copy()


    for _, B in B_all.groupby(starts[df["break_day"]]):
        idx = B.index

        # th should be constant within a break; take first non-null or default 0
        th_vals = B["th"].dropna()
        th = float(th_vals.iloc[0]) if not th_vals.empty else 0.0

        last_nc = None       # last day with nc_day==True inside this break
        last_any = None      # last day with any event inside this break (nc_day True inside break)

        for d in idx:
            # Safety: if coding leaked into a break, treat as ACTIVE
            if df.at[d, "coding_day"]:
                state.at[d] = "ACTIVE"
                last_any = d
                continue

            if df.at[d, "nc_day"]:
                state.at[d] = "NON_CODING"
                last_nc = d
                last_any = d
                continue

            # No nc event this day: evaluate GONE first
            days_since_any = util.daysBetween(d, (last_any if last_any is not None else idx[0]))
            if days_since_any >= cfg.gone_threshold:
                state.at[d] = "GONE"
                continue

            # Then NON_CODING vs INACTIVE based on Tfov
            gap_ref = last_nc if last_nc is not None else idx[0]
            gap = util.daysBetween(d, gap_ref)
            if gap > th:
                state.at[d] = "INACTIVE"
            else:
                state.at[d] = "NON_CODING"

    # 4) Transitions (only when the label changes day-to-day)
    transitions = []
    prev = None
    for d in df.index:
        cur = state.at[d]
        t = ""
        if prev is not None and cur != prev:
            if prev == "NON_CODING" and cur == "ACTIVE":
                t = "BACK_TO_CODING"
            elif prev == "INACTIVE" and cur in ("ACTIVE", "NON_CODING"):
                t = "REACTIVATION"
            elif prev == "GONE" and cur in ("ACTIVE", "NON_CODING"):
                t = "COMEBACK"
        transitions.append(t)
        prev = cur

    df["state"] = state.values
    df["transition"] = transitions
    return df

label_developers_activity()

Start Identifying inactivity periods for Rdatatable/data.table...
6 Developers inactivity periods identified
mattdowle


C:\Users\samut\AppData\Local\Temp\ipykernel_15848\905234148.py:109: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)


MichaelChirico


C:\Users\samut\AppData\Local\Temp\ipykernel_15848\905234148.py:109: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)


jangorecki


C:\Users\samut\AppData\Local\Temp\ipykernel_15848\905234148.py:109: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)
C:\Users\samut\AppData\Local\Temp\ipykernel_15848\905234148.py:109: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)


arunsrinivasan
ben-schwen
tdhock


C:\Users\samut\AppData\Local\Temp\ipykernel_15848\905234148.py:109: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)
C:\Users\samut\AppData\Local\Temp\ipykernel_15848\905234148.py:109: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(bool)


Algo A (4-state, no NCUT) → “Pure daily state machine.”
Always emits one of ACTIVE / NON_CODING / INACTIVE / GONE per day from activity signals alone; breaks are optional gates (for INACTIVE/GONE only). It respects non-coding activity anywhere on the timeline.

In [ ]:
ST_ACTIVE = "ACTIVE"
ST_NC     = "NON_CODING"
ST_INA    = "INACTIVE"
ST_GONE   = "GONE"

def label_developers_activity() -> pandas.DataFrame:
    """
    main function for labeling developers

    sets up varables to call label_timeline

    """
    #identifyInactivityPeriods
    #"Resources/repositories.txt"
    repos_file= '../' + cfg.repos_file
    #"../Organizations"
    organizationFolder = cfg.main_folder


    #identifyBreaks
    win = cfg.sliding_window_size
    shift = cfg.shift

    with open(repos_file) as f:
        repos_line = f.readlines()
        for repo in repos_line:
            #take the end '\n' out
            repo = repo.rstrip('\n')
            organization, project = repo.split('/')

            print(f"Start Identifying inactivity periods for {organization}/{project}...")

            authors, pauses = identifyInactivityPeriods( organizationFolder, organization, project)
            #make pauses to a csv file at this location C:\Users\samut\OneDrive\Documents\GitHub\developersInactivityAnalysisCOPY\Organizations\Rdatatable\data.table\Results
        
            pauses_list = pauses.values.tolist()
            print(f"{len(authors)} Developers inactivity periods identified")

            output_folder = organizationFolder + '/' + repo + "/Results"
            os.makedirs(output_folder, exist_ok=True)
            
            for dev in authors:
                timeline_folder = organizationFolder + '/' + repo + '/' + cfg.timeline_folder_name
                os.makedirs(timeline_folder, exist_ok=True)
            
                timeline_path = Path(timeline_folder) / f"{dev}_timeline.csv"

                if timeline_path.is_file():
                    user_timeline = pandas.read_csv(timeline_path, sep=cfg.CSV_separator, index_col=0)
                else:
                    folder = organizationFolder + '/' + repo
                    user_timeline = get_timeline(folder, dev)

                    #transpose the user_timeline making the frist row the first column
                    user_timeline.to_csv(timeline_path, sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, quoting=None, lineterminator='\n')

                print(f"{dev}")
            
                #make a break folder 
                breaks_folder = organizationFolder + '/' + repo + "/Breaks"
                os.makedirs(breaks_folder, exist_ok=True)
        
                breaks_path =  Path(breaks_folder)/  f"{dev}_breaks.csv"              

                if breaks_path.is_file():
                    breaks_df = pandas.read_csv(breaks_path, sep=cfg.CSV_separator, index_col=0)  

                else:
                    breaks_df = pandas.DataFrame(columns=['len', 'dates', 'th'])
                    breaks_df = identifyBreaks(pauses_list, dev=dev, window=win, shift=shift, debug_folder=output_folder )
                    breaks_df.to_csv(breaks_path, sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, index=False, lineterminator="\n")
                            
                #add label_timeline
                user_timeline = label_timeline(user_timeline, breaks_df)


                out_csv = Path(output_folder) / f"{dev}_labeled_timeline.csv"
                user_timeline.to_csv(out_csv,
                                      sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, quoting=None, lineterminator='\n')

def _parse_breaks_to_intervals(breaks_df: pandas.DataFrame, include_edges: bool) -> list[tuple[pandas.Timestamp,pandas.Timestamp,float,int]]:
    """Return list of (start,end,th,len). Edges trimmed unless include_edges=True."""
    ivals = []
    if breaks_df is None or breaks_df.empty:
        return ivals
    for r in breaks_df.itertuples(index=False):
        a, b = r.dates.split("/")
        start = pandas.to_datetime(a)
        end   = pandas.to_datetime(b)
        if not include_edges:
            start = start + pandas.Timedelta(days=1)
            end   = end   - pandas.Timedelta(days=1)
        if start <= end:
            th_val  = float(getattr(r, "th", np.nan)) if "th" in breaks_df.columns else np.nan
            len_val = int(getattr(r, "len", -1))      if "len" in breaks_df.columns else -1
            ivals.append((start.normalize(), end.normalize(), th_val, len_val))
    return ivals

def _apply_break_mask(df: pandas.DataFrame, breaks_df: pandas.DataFrame, include_break_edges: bool):
    """Stamp break_day/th/len onto df (overwrites any partially filled values)."""
    df["break_day"] = False
    if "th"  not in df.columns:  df["th"]  = np.nan
    if "len" not in df.columns:  df["len"] = np.nan
    ivals = _parse_breaks_to_intervals(breaks_df, include_break_edges)
    for s, e, th_val, len_val in ivals:
        rng = pandas.date_range(s, e, freq="D")
        inter = df.index.intersection(rng)
        if len(inter):
            df.loc[inter, "break_day"] = True
            if not np.isnan(th_val):
                df.loc[inter, "th"] = th_val
            if len_val >= 0:
                df.loc[inter, "len"] = float(len_val)
    # types
    df["break_day"] = df["break_day"].fillna(False).astype(bool)

def _transition_type(prev_state: str, state: str) -> str:
    if prev_state == ST_NC and state == ST_ACTIVE:
        return "back_to_coding"
    if prev_state == ST_INA and state in (ST_ACTIVE, ST_NC):
        return "reactivation"
    if prev_state == ST_GONE and state in (ST_ACTIVE, ST_NC):
        return "comeback"
    return "other"

def label_timeline(
    user_timeline: pandas.DataFrame,
    breaks_df: pandas.DataFrame,
    *,
    include_break_edges: bool = False,
    gate_inactive_by_breaks: bool = True,
    tfov_days: int | None = None,
    gone_days: int = 365,
    start_state: str = ST_INA,
    add_transitions: bool = True
) -> pandas.DataFrame:
    """
    Inputs
    ------
    user_timeline: index 'date' daily, columns at least:
        commits,pull_requests,issues,issues_comments,issues_events,pull_requests_comments,
        coding_day(bool),nc_day(bool)
        (If coding_day/nc_day missing, we derive them.)
    breaks_df: columns ['len','dates','th'] (legacy breaks)

    Outputs (added columns)
    -----------------------
    state ∈ {ACTIVE, NON_CODING, INACTIVE, GONE}
    silent_days (days since last coding OR non-coding event; -1 if unknown)
    break_day (bool), th, len   # re-stamped consistently
    transition (prev→curr) and transition_type (optional)
    """
    df = user_timeline.copy()

    # Ensure index is daily DatetimeIndex
    if "date" in df.columns:
        df = df.set_index("date")
    if not isinstance(df.index, pandas.DatetimeIndex):
        df.index = pandas.to_datetime(df.index)
    df.index = df.index.normalize()
    df = df.sort_index()
    # Ensure no missing days in between
    full_idx = pandas.date_range(df.index.min(), df.index.max(), freq="D")
    df = df.reindex(full_idx)

    # Build coding_day / nc_day if absent
    if "coding_day" not in df.columns:
        df["coding_day"] = (df.get("commits", 0).fillna(0) > 0) | (df.get("pull_requests", 0).fillna(0) > 0)
    if "nc_day" not in df.columns:
        other = (
            df.get("issues", 0).fillna(0)
            + df.get("issues_comments", 0).fillna(0)
            + df.get("issues_events", 0).fillna(0)
            + df.get("pull_requests_comments", 0).fillna(0)
        )
        df["nc_day"] = other > 0
    df["coding_day"] = df["coding_day"].fillna(False).astype(bool)
    df["nc_day"]     = df["nc_day"].fillna(False).astype(bool)

    # (Re)stamp break_day / th / len from breaks_df to be consistent, and (optionally) trim edges
    _apply_break_mask(df, breaks_df, include_break_edges=include_break_edges)

    # Pick tfov_days: prefer daily 'th' median if present; else breaks_df median; else default 30
    if tfov_days is None:
        if "th" in df.columns and df["th"].notna().any():
            tfov_days = int(round(df["th"].dropna().median()))
        elif breaks_df is not None and "th" in breaks_df.columns and breaks_df["th"].notna().any():
            tfov_days = int(round(breaks_df["th"].dropna().median()))
        else:
            tfov_days = 30

    # Convenience counts for output
    df["coding_count"]    = df.get("commits", 0).fillna(0) + df.get("pull_requests", 0).fillna(0)
    df["noncoding_count"] = (
        df.get("issues", 0).fillna(0)
        + df.get("issues_comments", 0).fillna(0)
        + df.get("issues_events", 0).fillna(0)
        + df.get("pull_requests_comments", 0).fillna(0)
    )

    # State machine
    states, silent_days, transitions, ttypes = [], [], [], []
    state = start_state
    prev_state = state
    last_event = None  # last day with coding OR non-coding

    for d, row in df.iterrows():
        coding = bool(row["coding_day"])
        nc     = bool(row["nc_day"])
        in_break = bool(row["break_day"])

        if coding:
            state = ST_ACTIVE
            last_event = d
            silent = 0
        elif nc:
            state = ST_NC
            last_event = d
            silent = 0
        else:
            # silent day
            if last_event is None:
                # Pre-history: keep whatever start_state you set, don't force GONE/INA
                silent = -1
            else:
                silent = (d - last_event).days

            # Only allow INACTIVE/GONE transitions when we are inside an algorithmic break window,
            # if gate_inactive_by_breaks=True
            can_downgrade = (not gate_inactive_by_breaks) or in_break

            if can_downgrade and last_event is not None:
                if silent > gone_days:
                    state = ST_GONE
                elif silent > tfov_days:
                    state = ST_INA
                else:
                    # remain in previous state (ACTIVE or NON_CODING)
                    state = state
            else:
                # remain in previous state (ACTIVE or NON_CODING)
                state = state

        states.append(state)
        silent_days.append(silent)

        if add_transitions:
            tr = f"{prev_state}->{state}" if state != prev_state else ""
            transitions.append(tr)
            ttypes.append(_transition_type(prev_state, state) if tr else "")
            prev_state = state

    df["state"] = states
    df["silent_days"] = silent_days
    if add_transitions:
        df["transition"] = transitions
        df["transition_type"] = ttypes

    return df

label_developers_activity()

# TESTER FOR BREAK IDENTIFICATION


In [ ]:
def write_pauses_table(
        df: pandas.DataFrame,
        out_path: os.PathLike,
        authors: list[str] | None = None,
        *,
        user_col: str = "author_id",
        date_col: str = "created_at",
        tail_to_today: bool = False
    ) -> pandas.DataFrame:

    df[date_col] = pandas.to_datetime(df[date_col]).dt.normalize()

    if authors is None:
        authors = df[user_col].unique()

    rows = []
    
    count =0
    for dev in authors:
        user_df = df[df[user_col] == dev]
        pause_len_1 = len(user_df[date_col].dt.date.unique())
        pause_len_2 =len(user_df)
        if user_df.empty:
            continue

        active_days = sorted(user_df[date_col].dt.date.unique())
        current_row = [dev]

        for i in range(len(active_days) - 1):
            prev_day = active_days[i]
            next_day = active_days[i + 1]
            gap = (next_day - prev_day).days
            if gap > 1:
                # Inactivity starts the day after prev_day
                current_row.append(f"{(prev_day + pandas.Timedelta(days=1)).strftime('%Y-%m-%d')}/{next_day.strftime('%Y-%m-%d')}")
            else:
                count += 1


        if tail_to_today and active_days:
            today = _date.today()
            gap = (today - active_days[-1]).days
            if gap > 1:
                current_row.append(f"{active_days[-1]}/{today}")

        if len(current_row) > 1:
            rows.append(current_row)
    
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", newline="",encoding="utf-8" ) as f:
        csv.writer(f, delimiter=",", quoting=csv.QUOTE_NONE).writerows(rows)

    return pandas.DataFrame(rows)

def get_commit_based_core_devs(commits, threshold=0.8):
    """
    commits: List[dict] where each dict contains at least the 'author' key.
    Example: [{'author': 'alice'}, {'author': 'bob'}, {'author': 'alice'}, ...]

    Returns: List of core developers (author names) who together authored >= threshold of commits.
    """
    # Count commits per developer
    author_commit_counts = Counter(commit["author_id"] for commit in commits)

    # Sort developers by number of commits (descending)
    sorted_authors = author_commit_counts.most_common()

    total_commits = sum(author_commit_counts.values())
    cumulative = 0
    core_devs = []

    for author, count in sorted_authors:
        cumulative += count
        core_devs.append(author)
        if cumulative / total_commits >= threshold:
            break

    return core_devs

def identifyInactivityPeriods(organizationFolder, organization, project):
    """Identifies the inactivity periods of the developers in the organization"""
    #url = "https://github.com/" + organization + "/" + project + ".git"
    #authors, emails = findCoreDevelopers(url, name=project)
    
    organizationFolder = organizationFolder + "/" + organization + "/" + project

    commits =  pandas.read_csv(organizationFolder + "/commit_list.csv", parse_dates=["created_at"], encoding="utf-8", header=0, sep=cfg.CSV_separator)

    commit_authors = get_commit_based_core_devs(commits.to_dict(orient='records'))

    pauses = write_pauses_table(commits, organizationFolder + "/pauses_commits.csv", commit_authors, user_col = "author_id", date_col="created_at")
    print("pauses", pauses)
    print("commits", commits)
    return commit_authors, pauses
repos_file= '../' + cfg.repos_file
#"../Organizations"
organizationFolder = cfg.main_folder
with open(repos_file) as f:
    repos_file = f.readlines()
    for repo in repos_file:
        #take the end '\n' out
        repo = repo.rstrip('\n')
        organization, project = repo.split('/')

        print(f"Start Identifying inactivity periods for {organization}/{project}...")

        authors, pauses = identifyInactivityPeriods( organizationFolder, organization, project)



In [3]:
def getFarOutThreshold(values, dev): ### If it is satisfying, move the function into UTILITIES
    import numpy
    th = 0
    q_3rd = numpy.percentile(values,75)
    q_1st = numpy.percentile(values,25)
    iqr = q_3rd-q_1st
    if iqr > 1:
        th = q_3rd + 3*iqr
    return th

def addToBreaksList(pauses, currentBreaks, th):
    for _, p in pauses.iterrows():
        if (p['len'] > th) and (p['dates'] not in currentBreaks.dates.tolist()):
            util.add(currentBreaks, [p['len'], p['dates'], th])
    return currentBreaks

def cleanClearBreaks(clearBreaks, breaks):
    for _, b in breaks.iterrows():
        clearBreaks = clearBreaks[clearBreaks.dates != b['dates']] # If it was in the long_breaks list, remove ot from there
    return clearBreaks

def identifyBreaks(pauses_dates_list, dev, window, shift,
                   debug_folder=None):           # NEW ARG
    '''
    Removes SURE BREAKS from windows to calculate Tfov
    and — with debug_folder — writes a per-window diagnostics CSV.
    '''
    breaks_df = pandas.DataFrame(columns=['len', 'dates', 'th'])
    diagnostics = []                             # NEW
    count = 0
    for row in pauses_dates_list:
        if row[0] != dev:              # ⬅️  ignore other developers
            continue
        
        count += 1
        if count % 50 == 0:  # Print progress every 100 developers
            print(count)
        intervals_list = [ x for x in row[1:]
                          if isinstance(x, str) and '/' in x and x.strip()]
        
        intervals_list.sort(key=lambda s: s.split('/')[0])

        if not all(a.split('/')[0] <= b.split('/')[0]
                for a, b in zip(intervals_list, intervals_list[1:])):
            print("⚠️  intervals_list UNSORTED for", dev)

        if not intervals_list:
            print(dev, 'has NO valid pauses')
            continue                      # <- don’t bail out; just skip

        clear_breaks = pandas.DataFrame(columns=['len', 'dates'])

        FPS_dt = datetime.strptime(intervals_list[0].split('/')[0], '%Y-%m-%d')
        LPE_dt = datetime.strptime(intervals_list[-1].split('/')[1], '%Y-%m-%d')

        win_start, win_end = FPS_dt, FPS_dt + timedelta(days=window)
        last_th = 0
        while win_end < LPE_dt:
            win_pauses_list = pandas.DataFrame(columns=['len', 'dates'])
            partially_included_pauses_list = pandas.DataFrame(columns=['len', 'dates'])

            for interval in intervals_list:
                int_start_str, int_end_str = interval.split('/')          # keep strings
                int_start_dt  = datetime.strptime(int_start_str, '%Y-%m-%d')
                int_end_dt    = datetime.strptime(int_end_str,   '%Y-%m-%d')
                pause_len = util.daysBetween(int_start_str, int_end_str)
                # fully inside
                if int_start_dt >= win_start and int_end_dt <= win_end:
                    util.add(win_pauses_list, [pause_len, interval])
                # touches boundary
                if ((int_start_dt <= win_end and int_end_dt > win_end) or
                    (int_end_dt >= win_start and int_start_dt < win_start)):
                    util.add(partially_included_pauses_list, [pause_len, interval])

            win_pauses = len(win_pauses_list)
            pauses = pandas.concat([win_pauses_list,
                                    partially_included_pauses_list],
                                    ignore_index=True)

            # --- decision logic (unchanged) ---------------------------------
            win_th = None
            added_flag = False
            if win_pauses >= 4:
                win_th = getFarOutThreshold(win_pauses_list['len'], dev)
                if win_th > 0:
                    before = len(breaks_df)
                    breaks_df = addToBreaksList(pauses, breaks_df, win_th)
                    added_flag = len(breaks_df) > before
                    last_th = win_th
                elif last_th > 0:
                    before = len(breaks_df)
                    breaks_df = addToBreaksList(pauses, breaks_df, last_th)
                    added_flag = len(breaks_df) > before
            else:
                if last_th > 0:
                    before = len(breaks_df)
                    breaks_df = addToBreaksList(pauses, breaks_df, last_th)
                    added_flag = len(breaks_df) > before

                clear_breaks = cleanClearBreaks(clear_breaks, breaks_df)
                for _, p in pauses.iterrows():
                    if (p['len'] >= window and
                        p['dates'] not in clear_breaks.dates.tolist() and
                        p['dates'] not in breaks_df.dates.tolist()):
                        util.add(clear_breaks, p)

            # ----------- NEW: record diagnostics for this window -------------
            diagnostics.append({
                'win_start': win_start.date(),
                'win_end':   win_end.date(),
                'win_pauses': win_pauses,
                'pause_lengths': ';'.join(map(str, win_pauses_list['len'].tolist())),
                'partial_lengths': ';'.join(map(str, partially_included_pauses_list['len'].tolist())),
                'win_th': win_th,
                'last_th': last_th,
                'added_as_break': 'yes' if added_flag else 'no'
            })
            # -----------------------------------------------------------------

            win_start += timedelta(days=shift)
            win_end   = win_start + timedelta(days=window)


    return breaks_df


In [12]:
def _load_activity_csv(folder: str,
                       filename: str,
                       rename_map: Dict[str, str],
                       dev_login,
                       usecols: list[str] = None,
                       ) -> pandas.DataFrame:
    """
    Read *filename* in *folder*, rename to the canonical columns
    ('id','date','creator_login'), keep ONLY the specified dev, and
    return three columns.  On any problem → empty df.
    """
    path = os.path.join(folder, filename)
    try:
        df = pandas.read_csv(path, sep=cfg.CSV_separator, usecols=usecols)
    except FileNotFoundError:
        logging.info("File %s not found – skipping", path)
        return pandas.DataFrame(columns=["id", "date", "creator_login"])
    except Exception as e:
        logging.warning("Could not read %s: %s", path, e)
        return pandas.DataFrame(columns=["id", "date", "creator_login"])

    df = df.rename(columns=rename_map)
    # keep only the columns we need, ignore anything extra
    df = df[["id", "date", "creator_login"]]
    df = df[df.creator_login == dev_login]
    # allow str OR list[str]
    if isinstance(dev_login, list):
        df = df[df.creator_login.isin(dev_login)]
    else:
        df = df[df.creator_login == dev_login]
    return df.reset_index(drop=True)

def get_activities(folder: str, dev_login: str) -> pandas.DataFrame:
    """
    Build the developer's DAILY 'other-actions' table.
    Returns a dataframe whose index is the *action*
    ('issues/pull_requests', 'issues_comments', …) and whose
    columns are day-strings.
    """
    files = {
    "prs": (
        "prs_repo.csv",
        {"PR_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "prs_comments": (
        "prs_comments.csv",
        {"comment_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues": (
        "issues_repo.csv",
        {"issue_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues_comments": (
        "issues_comments_repo.csv",
        {"comment_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues_events": (
        "issues_events_repo.csv",
        {"event_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues_timeline": (
        "issues_timeline_repo.csv",
        {"event_id": "id", "created_at": "date", "created_by": "creator_login"},
    )
    }

    # ---------- read / filter every file ----------
    dfs = {}
    for key, (fname, rename_map) in files.items():
        dfs[key] = _load_activity_csv(folder, fname, rename_map, dev_login)

    # ---------- split issues vs PRs -------------
    # Old logic: issues endpoint also returns PRs; remove rows whose id
    # matches a PR id so we don’t double-count.
    if not dfs["issues"].empty and not dfs["prs"].empty:
        dfs["issues"] = dfs["issues"][~dfs["issues"].id.isin(dfs["prs"].id)]

    # ---------- build the day range -------------
    # Derive it from the *actual* activity we just read.
    #
    # 1) gather every non-empty dataframe
    non_empty = [df for df in dfs.values() if not df.empty]

    if non_empty:
        # 2) earliest / latest date across *all* action types
        min_date = min(df["date"].min() for df in non_empty)
        max_date = max(df["date"].max() for df in non_empty)
    else:
        # Developer has no activity at all → default to one-day range
        min_date = max_date = pandas.Timestamp.today()

    # 3) full, dense list of day strings
    day_cols = (
        pandas.date_range(
            start=pandas.to_datetime(min_date).normalize(),
            end=pandas.to_datetime(max_date).normalize(),
            freq="D",
        )
        .strftime("%Y-%m-%d")
        .tolist()
    )

    # ---------- helper to create one timeline row ----------
    def _timeline_row(action_name, df_raw):
        row = [action_name]
        if df_raw.empty:
            row += [0] * len(day_cols)
            return row
        counts = (
            pandas.to_datetime(df_raw["date"])
            .dt.date
            .value_counts()
            .to_dict()
        )
        for d in day_cols:
            row.append(counts.get(pandas.to_datetime(d).date(), 0))
        return row

    # ---------- compile all action rows ----------
    rows = []
    if not dfs["issues"].empty:
        rows.append(_timeline_row("issues", dfs["issues"]))
    if not dfs["issues_comments"].empty:
        rows.append(_timeline_row("issues_comments", dfs["issues_comments"]))
    if not dfs["issues_events"].empty:
        rows.append(_timeline_row("issues_events", dfs["issues_events"]))
    if not dfs["prs"].empty:
        rows.append(_timeline_row("pull_requests", dfs["prs"]))
    if not dfs["prs_comments"].empty:
        rows.append(_timeline_row("pull_requests_comments", dfs["prs_comments"]))

    # (commits are already encoded in coding_history_table, so we skip them here)

    actions = pandas.DataFrame(rows, columns=["action"] + day_cols).set_index("action")

    return actions

def finalStep( period_detail, action_days, th, status, previously):
    last_idx = len(period_detail) - 1
    if last_idx < 0:
        return period_detail

    last_end = period_detail.at[last_idx, 'dates'].split('/')[1]
    if status == 'NCUT':
        if last_end == cfg.data_collection_date:
            status = cfg.NC
            start = period_detail.at[last_idx, 'dates'].split('/')[1]
            size = util.daysBetween(start, last_end)
            util.add(period_detail, [size, start+'/'+last_end, th, status, previously])
        else:
            status = previously
            break_start = period_detail.at[last_idx, 'dates'].split('/')[0]
            new_end = action_days[ 1]
            period_detail.at[last_idx, 'len']  = util.daysBetween(break_start, new_end)
            period_detail.at[last_idx, 'dates'] = f'{break_start}/{new_end}'
            last_end = new_end
            # Same th
            # Same status
            # Same previously
    if last_end == cfg.data_collection_date:
        period_detail.at[last_idx, 'label'] += '(NOW)'
    return period_detail

def handle_non_coding(current_status , i, th, period_detail, size, residual, action_days):
    previously = current_status 
    if size > th:
        next_status   = BreakGoneCheck(residual)

        dates = action_days[i] + '/' + action_days[i + 1]
        util.add(period_detail, [size, dates, th, next_status , previously])
        return next_status  
    else:
        break_start  = period_detail.at[len(period_detail)-1, 'dates'].split('/')[0]
        new_end = (datetime.strptime(action_days[i+1], "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")
        period_detail.at[len(period_detail)-1, 'dates'] = f'{break_start }/{new_end}'
        period_detail.at[len(period_detail)-1, 'len']  = util.daysBetween(break_start , new_end)
        return 'NON_CODING'

def handle_inactive(current_status: str,
                    i: int,
                    th: int,
                    period_detail: pandas.DataFrame,
                    size: int,
                    residual: int,
                    action_days: list[str]):

    start = action_days[i]          # first day with NO activity
    end   = action_days[i + 1]      # first day WITH activity (exclusive)
    previously = current_status

    # 1)  Short gap  →  NCUT  -------------------------------------------
    if size < th:
        status = 'NCUT'
        return status, start        # NCUT lasts the whole gap
    

    # 2)  Long gap – at least `th` days  -------------------------------
    #     First `th` days are NON_CODING
    nc_end_dt = (datetime.strptime(start, "%Y-%m-%d")
                 + timedelta(days=th - 1))     # inclusive range
    nc_end = nc_end_dt.strftime("%Y-%m-%d")

    util.add(period_detail,
             [th,
              f"{start}/{nc_end}",
              th,
              'NON_CODING',
              previously])

    # Anything left after those `th` days becomes an INACTIVE break
    residual = size - th
    if residual > 0:
        inactive_start_dt = nc_end_dt + timedelta(days=1)
        inactive_start = inactive_start_dt.strftime("%Y-%m-%d")

        util.add(period_detail,
                 [residual,
                  f"{inactive_start}/{end}",
                  th,
                  'INACTIVE',
                  'NON_CODING'])

        status = 'INACTIVE'
        period_start = inactive_start
    else:
        # Gap ended exactly on the threshold – we stop at NON_CODING
        status = 'NON_CODING'
        period_start = nc_end

    return status, period_start

def handle_active(curr, i, th, df, size, residual, days):
    df_end   = (datetime.strptime(days[i+1], "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")
    
    
    if days[i] == df_end:          # 0-day gap → skip
            return curr, None
    
    gap_len = util.daysBetween(days[i], df_end)

    if size <= th:
        df_start = days[i]
        df_end   = (datetime.strptime(days[i+1], "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")
        util.add(df, [gap_len,
                      f'{days[i]}/{df_end}',
                      0, 'ACTIVE', curr])
        return 'ACTIVE', None            # stay active
    
    next_status = BreakGoneCheck(residual)
    util.add(df, [gap_len,
                  f'{days[i]}/{df_end}',
                  th, next_status, curr])
    return next_status, None

def handle_ncut(period_start, i, th, period_detail,
                action_days):

    end_of_gap = action_days[i + 1]

    gap_len = util.daysBetween(period_start, end_of_gap)

    if gap_len <= th:
        return 'NCUT', period_start


    final_date = (datetime.strptime(period_start, "%Y-%m-%d")
                  + timedelta(days=th + 1)).strftime("%Y-%m-%d")

    util.add(period_detail,
             [util.daysBetween(period_start, final_date),
              f'{period_start}/{final_date}',
              th,
              'NON_CODING',
              'NCUT'])

    residual_len = util.daysBetween(final_date, end_of_gap)
    next_status  = BreakGoneCheck(residual_len)

    util.add(period_detail,
             [residual_len,
              f'{final_date}/{end_of_gap}',
              th,
              next_status,
              'NON_CODING'])

    return next_status, None            # NCUT spell is closed
 # Return the new status and the next period start date

def BreakGoneCheck(residual):
    if residual > cfg.gone_threshold:
        status = 'GONE'
    else:
        status = 'INACTIVE'

    return status

def splitBreak(break_limits, action_days, th):
    status = 'ACTIVE'  # NCUT: Non coding under threshold.
    previously = status
    period_start = ''

    #sort the action_days to ensure they are in chronological order
    break_range = break_limits.split('/')

    if break_range[0] not in action_days:       # first day of the break
        action_days.insert(0, break_range[0])
    if break_range[1] not in action_days:       # last day of the break
        action_days.append(break_range[1])

    action_days = sorted(set(action_days)) 

    period_detail = pandas.DataFrame(columns=['len', 'dates', 'th', 'label', 'previously'])


    for i in range(0, len(action_days) - 1):
        size = util.daysBetween(action_days[i], action_days[i + 1])
        if size == 0:
            continue 
        residual = size - (th + 1)

        if status == 'ACTIVE':
            status, period_start = handle_active(status , i, th, period_detail, size, residual, action_days)

        elif (status == 'INACTIVE') | (status == 'GONE'):
            status, period_start = handle_inactive(status, i, th, period_detail, size, residual, action_days)

        elif status == 'NON_CODING':
            status = handle_non_coding(status, i, th, period_detail, size, residual, action_days)

        elif status == 'NCUT':
            status, period_start = handle_ncut(period_start, i, th,
                                           period_detail, action_days)

    # A Final status 'INACTIVE', 'GONE' or 'NCUT' means an UNFREEZING ('NCUT' is not written into the detail list)
    period_detail = finalStep(period_detail, action_days, th, status, previously)

    return period_detail


In [ ]:
def main():
    #identifyInactivityPeriods
    #"Resources/repositories.txt"
    repos_file= '../' + cfg.repos_file
    #"../Organizations"
    organizationFolder = cfg.main_folder


    #identifyBreaks
    win = cfg.sliding_window_size
    shift = cfg.shift

    with open(repos_file) as f:
        repos_file = f.readlines()
        for repo in repos_file:
            #take the end '\n' out
            repo = repo.rstrip('\n')
            organization, project = repo.split('/')

            print(f"Start Identifying inactivity periods for {organization}/{project}...")

            authors, pauses = identifyInactivityPeriods( organizationFolder, organization, project)
            #make pauses to a csv file at this location C:\Users\samut\OneDrive\Documents\GitHub\developersInactivityAnalysisCOPY\Organizations\Rdatatable\data.table\Results
            pauses_list = pauses.values.tolist()
            print(f"{len(authors)} Devolpers inactivity periods identifyed")

            output_folder = organizationFolder + '/' + repo + "/Results"
            os.makedirs(output_folder, exist_ok=True)
            
            for dev in authors:
                print(f"{dev}")

                #make a break folder 
                breaks_folder = organizationFolder + '/' + repo + "/Breaks"
                os.makedirs(breaks_folder, exist_ok=True)
        
                breaks_path =  Path(breaks_folder)/  f"{dev}_breaks.csv"

                if breaks_path.is_file():
                    breaks_df = pandas.read_csv(breaks_path, sep=cfg.CSV_separator, index_col=0)  

                else:
                    breaks_df = pandas.DataFrame(columns=['len', 'dates', 'th'])
                    breaks_df = identifyBreaks(pauses_list, dev=dev, window=win, shift=shift, debug_folder=output_folder )
                    breaks_df.to_csv(breaks_path, sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, index=False, lineterminator="\n")
                
                actions_foler = organizationFolder + '/' + repo + '/' + cfg.actions_folder_name
                os.makedirs(actions_foler, exist_ok=True)
                actions_path = Path(actions_foler) / f"{dev}_actions_table.csv"

                if actions_path.is_file():
                    user_actions = pandas.read_csv(actions_path, sep=cfg.CSV_separator, index_col=0)
                else:
                    folder = organizationFolder + '/' + repo
                    user_actions = get_activities(folder, dev)
                    user_actions_t = user_actions.transpose() 
                    if dev == 'jangorecki':
                        user_actions_t.to_csv( Path(actions_foler) / f"{dev}_actions_table_transposed.csv", sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, quoting=None, lineterminator='\n')

                    #transpose the user_actions making the frist row the first column
                    user_actions.to_csv(actions_path, sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, quoting=None, lineterminator='\n')

                labeled_breaks = pandas.DataFrame(columns=['len', 'dates', 'th', 'label', 'previously'])
                # make a progress bar
                status = 'ACTIVE'
                end_of_last_break = None
                for _, b in breaks_df.iterrows():
                    # CHECK ACTIVITIES
                    
                    break_dates = b['dates']
                    threshold = b['th']
                    break_range = break_dates.split('/')
                    inner_start = (datetime.strptime(break_range[0], "%Y-%m-%d")).strftime("%Y-%m-%d")
                    inner_end = (datetime.strptime(break_range[1], "%Y-%m-%d")).strftime("%Y-%m-%d")

                    break_actions = user_actions.loc[:, inner_start:inner_end]  # Gets only the chosen period
                    break_actions = break_actions.loc[~(break_actions == 0).all(axis=1)]  # Removes the actions not performed

                    is_activity_day = (break_actions != 0).any()  # List Of Columns With at least a Non-Zero Value
                    action_days = is_activity_day.index[is_activity_day].tolist()  # List Of Columns NAMES Having Column Names at least a Non-Zero Value
                    
                    
                    if status:
                        previously = status
                    else:
                        previously = 'ACTIVE'
                        
                    #handle the time between breaks
                    if end_of_last_break is not None:

                        gap_end_dt =  (datetime.strptime(break_range[1], "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")
                        gap_start_dt = end_of_last_break  # Start of the gap is the day after the last break end


                        if gap_start_dt <= gap_end_dt:        # non-empty gap → ACTIVE row
                            gap_size = util.daysBetween(
                                gap_start_dt,
                                end_of_last_break
                            )
                            gap_row = pandas.DataFrame(
                                [[gap_size,
                                f"{gap_start_dt}/{end_of_last_break}",
                                0,             # th for ACTIVE is usually 0
                                'ACTIVE',
                                previously]],
                                columns=['len', 'dates', 'th', 'label', 'previously']  # 'source' is for debugging purposes
                            )
                            labeled_breaks = pandas.concat([labeled_breaks, gap_row],
                                                        ignore_index=True)

                    end_of_last_break = (datetime.strptime(break_range[1], "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")

                    if len(break_actions) > 0:  # There are other activities: the Break is Non-coding
                        break_detail = splitBreak(break_dates, action_days, threshold)
                        # Exclude columns where all entries are NA
                        labeled_breaks = labeled_breaks.dropna(axis=1, how='all')
                        break_detail = break_detail.dropna(axis=1, how='all')

                        status = labeled_breaks.at[len(labeled_breaks)-1, 'label'] if not labeled_breaks.empty else 'ACTIVE'
                        # Concatenate DataFrames
                        labeled_breaks = pandas.concat([labeled_breaks, break_detail], ignore_index=True)
                    else:  # No other activities: the Break is Inactive or Gone
                        size = util.daysBetween(inner_start, inner_end)
                        status = BreakGoneCheck(size)
                        dates  = f"{inner_start}/{inner_end}"


                        break_detail = pandas.DataFrame(
                            [[size, dates, threshold, status, previously]],  # ← shape matches util.add
                            columns=['len', 'dates', 'th', 'label', 'previously']  # 'source' is for debugging purposes
                        )
                        labeled_breaks = pandas.concat([labeled_breaks, break_detail], ignore_index=True) 

                        if break_duration > cfg.gone_threshold:
                            status = 'GONE'
                            previously = 'ACTIVE'
                            util.add(labeled_breaks, [break_duration, break_dates, threshold, status, previously])
                        else:
                            status = 'INACTIVE'
                            previously = 'ACTIVE'
                            util.add(labeled_breaks, [break_duration, break_dates, threshold, status, previously])

                        break_end = break_dates.split('/')[1]
                        if break_end == cfg.data_collection_date:
                            labeled_breaks.at[0, 'label'] += '(NOW)'
                        else:
                            util.add(labeled_breaks, [0, break_end, 0, 'ACTIVE', status])

                    


                #TO DO handle fianl break end to timestamp now.
                

                break_detail = pandas.DataFrame(
                            [[size, dates, threshold, status, previously]],  # ← shape matches util.add
                            columns=['len', 'dates', 'th', 'label', 'previously']  # 'source' is for debugging purposes
                        )
                labeled_breaks = pandas.concat([labeled_breaks, break_detail], ignore_index=True) 
   

                out_csv = Path(output_folder) / f"{dev}_labeled_breaks.csv"
                labeled_breaks.to_csv(out_csv,
                                      sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, index=False, quoting=None, lineterminator='\n')              
                
main()

Start Identifying inactivity periods for Rdatatable/data.table...
6 Devolpers inactivity periods identifyed
mattdowle
MichaelChirico
jangorecki
arunsrinivasan
ben-schwen
tdhock


In [14]:
def breaks_to_daily(breaks_df, *, user):

    daily_rows = []

    for _, row in breaks_df.iterrows():
        # Parse the date or date-range string
        if '/' in row["dates"]:
            start_str, end_str = row["dates"].split("/")
        else:                                      # single-day slice
            start_str = end_str = row["dates"]

        start = pandas.to_datetime(start_str).normalize()
        end   = pandas.to_datetime(end_str).normalize()

        # Build one entry per calendar day (inclusive)
        for day in pandas.date_range(start, end, freq="D"):
            daily_rows.append({
                "user":  row.get("user", user),   # prefer column, else arg
                "day":   day.date(),              # keep as python-date for clean csv
                "label": row["label"]
            })

    return (pandas.DataFrame(daily_rows)
              .sort_values("day")
              .reset_index(drop=True))


breaks = pandas.read_csv(r"C:\Users\samut\OneDrive\Documents\GitHub\developersInactivityAnalysisCOPY\Organizations\Rdatatable\data.table\Results\jangorecki_labeled_breaks.csv")

daily_timeline = breaks_to_daily(breaks, user="jangorecki")

daily_timeline.to_csv(r"C:\Users\samut\OneDrive\Documents\GitHub\developersInactivityAnalysisCOPY\Organizations\Rdatatable\data.table\Results\daily_timeline.csv",
                            sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, index=False, quoting=None, lineterminator='\n')


In [4]:
def get_activities(folder: str, dev_login: str) -> pandas.DataFrame:
    """
    Build a daily time-series of commit counts for `dev_login`.

    Parameters
    ----------
    folder     : str  – repo folder that contains commit_list.csv
    dev_login  : str  – GitHub login (author_id) of the developer

    Returns
    -------
    pd.DataFrame  – index = date (YYYY-MM-DD),
                    column 'commits' = # commits that day
    """
    path = os.path.join(folder, "commit_list.csv")

    # ─── load & clean ──────────────────────────────────────────────────
    df = pandas.read_csv(path, sep=cfg.CSV_separator, parse_dates=["created_at"])

    df = df[df["author_id"] == dev_login]           # keep only this dev
    if df.empty:                                    # no commits at all
        raise ValueError(f"No commits found for {dev_login}")

    # ─── per-day aggregation ──────────────────────────────────────────
    daily_counts = (
        df.groupby(df["created_at"].dt.normalize())       # strip hh:mm:ss
          .size()
          .rename("commits")
    )

    # ─── re-index so *every* day appears ──────────────────────────────
    full_index = pandas.date_range(daily_counts.index.min(),
                               daily_counts.index.max(),
                               freq="D")
    daily_counts = daily_counts.reindex(full_index, fill_value=0)

    daily_counts.index.name = "created_at"                # nice tidy index
    return daily_counts.to_frame()

get_activities("../Organizations/Rdatatable/data.table", "jangorecki").to_csv(
    "../Organizations/Rdatatable/data.table/Results/jangorecki_daily_commits.csv",
    sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, index=True, quoting=None, lineterminator='\n'
)

# Other

In [22]:
from github import Github

secrets = [

]
for new_token in secrets:
    ghub = Github(new_token)
    search_limit = ghub.get_rate_limit().search.remaining
    core_limit = ghub.get_rate_limit().core.remaining
    reset = ghub.get_rate_limit().core.reset
    #change the time to be in Mountain Standard Time (MST) 
    reset = reset.astimezone(tz=None)  # Convert to local timezone
    # Print the limits

    print(f"Search limit for token {new_token}:\n {search_limit}, {core_limit}, {reset} \n")


### Find Core Devs TF

In [3]:
#new
def findCoreDevelopers(
    url: str,
    dest_root: str | Path = ".tf_cache",
    *,
    name: str | None = None,
    branch: str | None = None,
    refresh: bool = False,
) -> tuple[list[str], list[str]]:        # <- correct annotation
    """
    Clone <url> (or reuse/refresh an existing clone) and run Truck‑Factor.
    Returns (authors, emails) – two parallel lists with the same length.
    """

    # ------------------------------------------------------------------ #
    # Paths
    # ------------------------------------------------------------------ #
    org, repo = url.rstrip("/").split("/")[-2:]
    repo = repo.removesuffix(".git")
    repo_path   = Path(cfg.main_folder) / org / repo          # actual repo
    cache_dir   = repo_path / dest_root                       # .tf_cache
    tf_csv      = cache_dir / "TruckFactor.csv"

    cache_dir.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------ #
    # 1. Return cached result if possible
    # ------------------------------------------------------------------ #
    if tf_csv.is_file() and not refresh:
        cache_df = pandas.read_csv(
            tf_csv,
            sep=cfg.CSV_separator,
            encoding="utf-8",
        )
        return (
            cache_df["login"].tolist(),
            cache_df["email"].tolist(),
        )

    # ------------------------------------------------------------------ #
    # 2. Ensure we have a local clone
    # ------------------------------------------------------------------ #
    try:
        if (cache_dir / ".git").is_dir():
            repo = Repo(cache_dir)
            if refresh:
                repo.git.fetch("--all", "--prune")
            if branch:
                repo.git.checkout(branch)
                if refresh:
                    repo.git.pull()
        else:
            # Empty dir or non‑existent – clone afresh
            if cache_dir.exists():
                shutil.rmtree(cache_dir, ignore_errors=True)
            repo = Repo.clone_from(url, cache_dir, branch=branch)
    except git_exc.GitCommandError as e:
        raise RuntimeError(f"Git failed: {e.stderr or e}") from e

    # ------------------------------------------------------------------ #
    # 3. Compute Truck Factor (this calls your patched compute_tf)
    # ------------------------------------------------------------------ #
    tf, critical_sha, authors, emails = compute_tf(str(cache_dir))

    # Always lists from here on
    authors = list(authors)
    emails  = list(emails)

    # ------------------------------------------------------------------ #
    # 4. Cache the result for next time
    # ------------------------------------------------------------------ #
    pandas.DataFrame({"login": authors, "email": emails}).to_csv(
        tf_csv,
        sep=cfg.CSV_separator,
        index=False,
        lineterminator="\n",
        encoding="utf-8",
    )

    return authors, emails
#old
def findCoreDevelopers(
    url: str,
    dest_root: str | Path = ".tf_cache",
    *,
    name: str | None = None,
    branch: str | None = None,
    refresh: bool = False,
) -> tuple[int, str, list[str]]:
    """
    Clone <url> (or reuse/refresh an existing clone) and run Truck-Factor.
    Returns (tf, critical_sha, authors).
    """
    # --------------------------------------------------------------------- #
    dest_root = Path(dest_root).expanduser().resolve()    # .../rails/rails
    name = name or url.rstrip("/").split("/")[-1].removesuffix(".git")
    dest = dest_root / name
    tf_cache  = dest / ".tf_cache"
    tf_cache.mkdir(parents=True, exist_ok=True)                  
    tf_csv = dest / "TruckFactor.csv"
    
    clone_path = dest / name                          # .../.tf_cache/rails

    if tf_csv.is_file():
        logging.info("TF cache hit – using %s", tf_csv)
        return pandas.read_csv(tf_csv, encoding="utf-8")["login"].tolist()

    

    repo = None
    # --------------------------------------------------------------------- #
    try:
        if clone_path.exists():
            try:
                repo = Repo(clone_path)
            except git_exc.InvalidGitRepositoryError:
                # Directory exists but isn't a repo – start fresh
                shutil.rmtree(dest, ignore_errors=True)
                repo = Repo.clone_from(url, to_path=dest, branch=branch)
            else:
                # Repo is valid – refresh if asked
                if refresh:
                    repo.git.fetch("--all", "--prune")
                if branch:
                    repo.git.checkout(branch)
                    if refresh:
                        repo.git.pull()
        else:
            repo = Repo.clone_from(url, clone_path, branch=branch)
    except git_exc.GitCommandError as e:
        raise RuntimeError(f"Git failed: {e.stderr or e}") from e

    # --------------------------------------------------------------------- #
    # Ensure the repo is NOT empty (at least one commit reachable)
    if not list(repo.iter_commits('--all', max_count=1)):
        # Something went wrong – start over with a clean clone
        shutil.rmtree(dest, ignore_errors=True)
        repo = Repo.clone_from(url, clone_path, branch=branch)

    # --------------------------------------------------------------------- #
    # Truck-Factor
    #if the truck factor file does not exist, we compute it
    if not tf_csv.is_file():
        print("Computing Truck Factor for", clone_path)
        tf, critical_sha, authors = compute_tf(str(clone_path))
        
    else:
        print("Using cached Truck Factor from %s", tf_csv)
    

    pandas.DataFrame(authors, columns=["login"])\
      .to_csv(tf_csv ,
              sep=cfg.CSV_separator,
              index=False,
              lineterminator="\n",
              encoding="utf-8")

    return authors